In [0]:
streaming_raw_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "raw/streaming/"
)

streaming_raw_df = spark.read.json(streaming_raw_path)

display(streaming_raw_df.limit(10))

data,event_type,ingestion_time,price,source,symbol,year,month,day,hour
"List(307.28, -0.98, -0.3179, 309.97, 307.0796, 308.04, 308.26, 1786455445)",stock_quote,2026-08-11T13:37:41.608566+00:00,null,finnhub,AAPL,2026,8,11,13
"List(306.84, -1.42, -0.4607, 309.97, 306.69, 308.04, 308.26, 1786455491)",stock_quote,2026-08-11T13:38:41.573249+00:00,null,finnhub,AAPL,2026,8,11,13
"List(307.37, -5.96, -1.9021, 310.01, 304.61, 310.01, 313.33, 1786391419)",stock_quote,2026-08-10T19:50:41.561836+00:00,null,finnhub,AAPL,2026,8,10,19
"List(307.02, -6.31, -2.0139, 310.01, 304.61, 310.01, 313.33, 1786391490)",stock_quote,2026-08-10T19:51:41.581473+00:00,null,finnhub,AAPL,2026,8,10,19
"List(307.02, -6.31, -2.0139, 310.01, 304.61, 310.01, 313.33, 1786391607)",stock_quote,2026-08-10T19:53:41.560921+00:00,null,finnhub,AAPL,2026,8,10,19
"List(307.24, -6.09, -1.9436, 310.01, 304.61, 310.01, 313.33, 1786391654)",stock_quote,2026-08-10T19:54:41.560494+00:00,null,finnhub,AAPL,2026,8,10,19
"List(308.26, -5.07, -1.6181, 308.26, 304.61, 306.83, 313.33, 1786392000)",stock_quote,2026-08-10T20:03:41.865899+00:00,null,finnhub,AAPL,2026,8,10,20
"List(308.26, -5.07, -1.6181, 308.26, 304.61, 306.83, 313.33, 1786392000)",stock_quote,2026-08-10T20:04:41.543011+00:00,null,finnhub,AAPL,2026,8,10,20
"List(308.26, -5.07, -1.6181, 308.26, 304.61, 306.83, 313.33, 1786392000)",stock_quote,2026-08-10T20:05:41.526183+00:00,null,finnhub,AAPL,2026,8,10,20
"List(308.26, -5.07, -1.6181, 308.26, 304.61, 306.83, 313.33, 1786392000)",stock_quote,2026-08-10T20:06:41.526773+00:00,null,finnhub,AAPL,2026,8,10,20


In [0]:
from datetime import datetime
from math import isfinite

from pyspark.sql import functions as F
from pyspark.sql.types import StringType


streaming_processed_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "processed/streaming/stock_quotes/"
)

streaming_quarantine_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "quarantine/streaming/stock_quotes/"
)

streaming_schema_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "checkpoints/streaming/schema/"
)

valid_checkpoint_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "checkpoints/streaming/valid_quotes/"
)

invalid_checkpoint_path = (
    "s3://stock-market-pipeline-mule-2026/"
    "checkpoints/streaming/invalid_quotes/"
)


def validate_quote(event_type, source, symbol, ingestion_time, price):
    if not all([event_type, source, symbol, ingestion_time]):
        return "missing_required_field"

    if event_type != "stock_quote":
        return "invalid_event_type"

    try:
        datetime.fromisoformat(str(ingestion_time).replace("Z", "+00:00"))
        price = float(price)
    except (ValueError, TypeError):
        return "invalid_value"

    if not isfinite(price) or price <= 0:
        return "invalid_price"

    return None


validate_quote_udf = F.udf(validate_quote, StringType())

raw_live_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", streaming_schema_path)
    .load(streaming_raw_path)
)

flat_live_stream = raw_live_stream.select(
    F.col("event_type"),
    F.col("source"),
    F.col("symbol"),
    F.col("ingestion_time"),
    F.get_json_object(F.col("data"), "$.c").alias("current_price"),
    F.get_json_object(F.col("data"), "$.d").alias("price_change"),
    F.get_json_object(F.col("data"), "$.dp").alias("percent_change"),
    F.get_json_object(F.col("data"), "$.h").alias("day_high"),
    F.get_json_object(F.col("data"), "$.l").alias("day_low"),
    F.get_json_object(F.col("data"), "$.o").alias("day_open"),
    F.get_json_object(F.col("data"), "$.pc").alias("previous_close"),
    F.get_json_object(F.col("data"), "$.t").alias("event_epoch"),
)

validated_live_stream = flat_live_stream.withColumn(
    "validation_error",
    validate_quote_udf(
        F.col("event_type"),
        F.col("source"),
        F.col("symbol"),
        F.col("ingestion_time"),
        F.col("current_price"),
    ),
)

valid_live_stream = (
    validated_live_stream
    .filter(F.col("validation_error").isNull())
    .select(
        F.col("symbol"),
        F.col("current_price").cast("double"),
        F.col("price_change").cast("double"),
        F.col("percent_change").cast("double"),
        F.col("day_high").cast("double"),
        F.col("day_low").cast("double"),
        F.col("day_open").cast("double"),
        F.col("previous_close").cast("double"),
        F.to_timestamp(F.col("ingestion_time")).alias("ingestion_time"),
        F.to_timestamp(F.from_unixtime(F.col("event_epoch"))).alias(
            "market_event_time"
        ),
        F.lit("finnhub").alias("source"),
        F.current_timestamp().alias("processed_at"),
    )
)

invalid_live_stream = (
    validated_live_stream
    .filter(F.col("validation_error").isNotNull())
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_query = (
    valid_live_stream.writeStream
    .format("parquet")
    .outputMode("append")
    .option("checkpointLocation", valid_checkpoint_path)
    .trigger(availableNow=True)
    .start(streaming_processed_path)
)

invalid_query = (
    invalid_live_stream.writeStream
    .format("parquet")
    .outputMode("append")
    .option("checkpointLocation", invalid_checkpoint_path)
    .trigger(availableNow=True)
    .start(streaming_quarantine_path)
)

valid_query.awaitTermination()
invalid_query.awaitTermination()

print("Streaming preprocessing complete.")
print(f"Clean Parquet data: {streaming_processed_path}")
print(f"Quarantined records: {streaming_quarantine_path}")

Streaming preprocessing complete.
Clean Parquet data: s3://stock-market-pipeline-mule-2026/processed/streaming/stock_quotes/
Quarantined records: s3://stock-market-pipeline-mule-2026/quarantine/streaming/stock_quotes/
